# Curating systems for SLiM simulations

In [4]:
import json
import os
import pandas as pd
import numpy as np
from tqdm import tqdm
import shutil

while os.path.split(os.getcwd())[-1] != 'calvados-slim':
    os.chdir('..')

from src import curation, utils

In [5]:
# Loading curations
with open('data/curation/systems.json') as file:
    systems = json.load(file)
systems_dir = os.path.abspath('data/curation/systems')

In [6]:
# Curating input files for systems
failing = []
for system in tqdm(systems):
    system_dir = os.path.join(systems_dir, system['ID'])
    if os.path.exists(system_dir):
        continue

    # Getting missing CIF files
    if not os.path.exists(system['SLiMs'][0]['PDB']):
        path, extension = os.path.splitext(system['SLiMs'][0]['PDB'])
        pdb_id = os.path.basename(path)
        if extension == '.cif' and utils.is_pdb(pdb_id):
            utils.download_cif(pdb_id, system['SLiMs'][0]['PDB'])

    try:
        curation.System.curate(system, output_dir=systems_dir)
    except Exception as e:
        print(f"Curation failed for system '{system['ID']}': {e}")
        shutil.rmtree(os.path.join(system_dir))
        failing.append([system['ID'], e])

100%|██████████| 278/278 [00:03<00:00, 86.93it/s]
